In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [3]:
# Load cleaned train and test datasets
train_df = pd.read_csv('../data/Industrial Risk and Reliability Assessment/cleaned_train_data.csv')
test_df = pd.read_csv('../data/Industrial Risk and Reliability Assessment/cleaned_test_data.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (61249, 27)
Test shape: (41214, 26)


In [4]:
# Define feature columns (exclude engine_id, cycle, and RUL)
feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

In [5]:
# Prepare X (features) and y (target - RUL)
X_train = train_df[feature_cols].values
y_train = train_df['RUL'].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (61249, 24)
y_train shape: (61249,)


In [6]:
# Define the ANN Model architecture
class RUL_ANN(nn.Module):
    def __init__(self, input_dim):
        super(RUL_ANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(32, 1)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output(x)
        return x

In [7]:
# Initialize model
input_dim = X_train.shape[1]
model = RUL_ANN(input_dim)
print(model)

RUL_ANN(
  (layer1): Linear(in_features=24, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (output): Linear(in_features=32, out_features=1, bias=True)
)


In [8]:
from torch.utils.data import TensorDataset, DataLoader

# Convert numpy arrays to PyTorch tensors
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

In [9]:
# Create TensorDataset and DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [10]:
# Loss function and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("DataLoader, Loss, and Optimizer are ready!")


DataLoader, Loss, and Optimizer are ready!


In [11]:
# Training Loop
epochs = 5
model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)
        
        # Calculate loss
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_X.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

print("ANN Model Training Completed Successfully!")

Epoch [1/5], Loss: 10488.4357
Epoch [2/5], Loss: 6727.7404
Epoch [3/5], Loss: 5169.9634
Epoch [4/5], Loss: 4331.5201
Epoch [5/5], Loss: 4086.8191
ANN Model Training Completed Successfully!


In [15]:
import os

# Create a new directory named 'saved_models'
os.makedirs('../saved_models', exist_ok=True)

# Save the model inside saved_models
torch.save(model.state_dict(), '../saved_models/rul_ann_model.pth')
print("ANN Model saved successfully inside 'saved_models' folder!")

ANN Model saved successfully inside 'saved_models' folder!
